# Results Analysis

Paste prediction CSV paths or URLs for the tactile, vision, and fusion detectors below, then run the notebook top to bottom.

The notebook reports metrics for every evaluated CSV over:

- `all`: every sample in the prediction CSV
- `visually_challenging`: rows whose `object_id` appears in `possibly-visually-difficult.csv`

Expected prediction columns: `sample_id`, `object_id`, `label`, `pred`, `prob_success`; `prob_failure` is optional.

In [ ]:
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

## 1. Configure Inputs

Use the same `comparison_id` for detector CSVs evaluated on the same samples. For example, use `overall_eval` for the three full-eval detector CSVs.

In [ ]:
RESULT_CSVS = {
    "vision": [
        # {"path": "reports/predictions/vision_eval.csv", "test_set": "eval", "comparison_id": "overall_eval"},
    ],
    "tactile": [
        # {"path": "reports/predictions/tactile_eval.csv", "test_set": "eval", "comparison_id": "overall_eval"},
    ],
    "fusion": [
        # {"path": "reports/predictions/fusion_eval.csv", "test_set": "eval", "comparison_id": "overall_eval"},
    ],
}

VISUALLY_DIFFICULT_CSV = Path("possibly-visually-difficult.csv")
REPORT_DIR = Path("reports/results_analysis")
LABEL_NAMES = ["failure", "success"]
POSITIVE_LABEL = 1
DECISION_THRESHOLD = 0.5
AGGREGATE_DUPLICATE_SAMPLE_IDS = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "metrics").mkdir(exist_ok=True)
(REPORT_DIR / "confusion_matrices").mkdir(exist_ok=True)
print(f"Writing analysis artifacts to {REPORT_DIR.resolve()}")

## 2. Load Predictions And Visually Challenging Object List

In [ ]:
def slugify(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9._-]+", "_", value)
    return value.strip("_") or "unnamed"


def spec_to_record(detector, spec, idx):
    if isinstance(spec, str):
        path = spec
        stem = Path(spec.split("?")[0]).stem
        return {
            "detector": detector,
            "path": path,
            "csv_name": stem or f"{detector}_{idx}",
            "test_set": stem or f"set_{idx}",
            "comparison_id": stem or f"set_{idx}",
        }
    record = dict(spec)
    record["detector"] = detector
    record["path"] = record.get("path") or record.get("url")
    if not record.get("path"):
        raise ValueError(f"Missing path/url for {detector} spec #{idx}: {spec}")
    stem = Path(str(record["path"]).split("?")[0]).stem
    record.setdefault("csv_name", stem or f"{detector}_{idx}")
    record.setdefault("test_set", record["csv_name"])
    record.setdefault("comparison_id", record["test_set"])
    return record


def normalize_label(series):
    if series.dtype == bool:
        return series.astype(int)
    mapped = series.replace({"False": 0, "True": 1, "failure": 0, "success": 1, "fail": 0, "pass": 1})
    return mapped.astype(int)


def load_prediction_csv(record):
    df = pd.read_csv(record["path"])
    required = {"sample_id", "object_id", "label", "pred", "prob_success"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{record['path']} is missing required columns: {sorted(missing)}")
    df = df.copy()
    df["label"] = normalize_label(df["label"])
    df["pred"] = normalize_label(df["pred"])
    df["prob_success"] = pd.to_numeric(df["prob_success"])
    df["prob_failure"] = pd.to_numeric(df["prob_failure"]) if "prob_failure" in df.columns else 1.0 - df["prob_success"]
    for key in ["detector", "csv_name", "test_set", "comparison_id", "path"]:
        df[key] = record[key]
    df["row_id"] = np.arange(len(df))
    return df


visual_source_df = pd.read_csv(VISUALLY_DIFFICULT_CSV)
if "object_name" not in visual_source_df.columns:
    raise ValueError(f"{VISUALLY_DIFFICULT_CSV} must contain an object_name column")
visually_challenging_objects = set(visual_source_df["object_name"].dropna().astype(str))
print(f"Loaded {len(visually_challenging_objects)} visually challenging objects from {VISUALLY_DIFFICULT_CSV}")

records = []
for detector, specs in RESULT_CSVS.items():
    for idx, spec in enumerate(specs):
        records.append(spec_to_record(detector, spec, idx))
if not records:
    raise ValueError("Add at least one CSV spec to RESULT_CSVS before running the notebook.")

raw_predictions = pd.concat([load_prediction_csv(record) for record in records], ignore_index=True)
raw_predictions["is_visually_challenging"] = raw_predictions["object_id"].isin(visually_challenging_objects)

display(raw_predictions.head())
display(raw_predictions.groupby(["comparison_id", "test_set", "detector", "csv_name"]).size().rename("rows").reset_index())
display(raw_predictions.groupby(["comparison_id", "detector"])["is_visually_challenging"].agg(samples="sum", fraction="mean").reset_index())

## 3. Aggregate Duplicate Sample IDs

If a CSV contains duplicate `sample_id` rows, for example tactile `--sensor both`, probabilities are averaged per sample before metrics are computed.

In [ ]:
def aggregate_duplicate_samples(df):
    group_cols = ["comparison_id", "test_set", "detector", "csv_name", "path", "sample_id"]
    object_mode = lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]
    out = (
        df.groupby(group_cols, as_index=False)
        .agg(
            object_id=("object_id", object_mode),
            label=("label", "first"),
            prob_success=("prob_success", "mean"),
            prob_failure=("prob_failure", "mean"),
            is_visually_challenging=("is_visually_challenging", "first"),
            source_rows=("row_id", "count"),
        )
    )
    out["pred"] = (out["prob_success"] >= DECISION_THRESHOLD).astype(int)
    return out


predictions = aggregate_duplicate_samples(raw_predictions) if AGGREGATE_DUPLICATE_SAMPLE_IDS else raw_predictions.copy()
display(predictions.groupby(["comparison_id", "detector"]).agg(samples=("sample_id", "nunique"), rows=("sample_id", "size")).reset_index())

all_subset = predictions.assign(subset="all")
visual_subset = predictions[predictions["is_visually_challenging"]].assign(subset="visually_challenging")
analysis_predictions = pd.concat([all_subset, visual_subset], ignore_index=True)
display(analysis_predictions.groupby(["comparison_id", "subset", "detector"]).size().rename("samples").reset_index())

## 4. Validate Sample Alignment

In [ ]:
alignment_rows = []
for (comparison_id, subset), group in analysis_predictions.groupby(["comparison_id", "subset"]):
    sample_sets = {det: set(det_group["sample_id"]) for det, det_group in group.groupby("detector")}
    union = set().union(*sample_sets.values()) if sample_sets else set()
    intersection = set.intersection(*sample_sets.values()) if sample_sets else set()
    for detector, samples in sample_sets.items():
        alignment_rows.append({
            "comparison_id": comparison_id,
            "subset": subset,
            "detector": detector,
            "n_samples": len(samples),
            "missing_from_detector": len(union - samples),
            "extra_vs_intersection": len(samples - intersection),
            "intersection_size": len(intersection),
            "union_size": len(union),
        })

alignment_df = pd.DataFrame(alignment_rows)
display(alignment_df)

## 5. Metrics And Confusion Matrices

In [ ]:
def safe_auroc(y_true, score):
    if len(pd.Series(y_true).dropna().unique()) < 2:
        return np.nan
    return roc_auc_score(y_true, score)


def metrics_for_group(group):
    y_true = group["label"].astype(int).to_numpy()
    y_pred = group["pred"].astype(int).to_numpy()
    score = group["prob_success"].astype(float).to_numpy()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    auroc = safe_auroc(y_true, score)
    return {
        "comparison_id": group["comparison_id"].iloc[0],
        "test_set": group["test_set"].iloc[0],
        "subset": group["subset"].iloc[0],
        "detector": group["detector"].iloc[0],
        "csv_name": group["csv_name"].iloc[0],
        "n": int(len(group)),
        "n_success": int((y_true == 1).sum()),
        "n_failure": int((y_true == 0).sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=[0, 1], zero_division=0)),
        "auroc": float(auroc) if not math.isnan(auroc) else np.nan,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "confusion_matrix": cm.tolist(),
    }


metrics = []
for _, group in analysis_predictions.groupby(["comparison_id", "subset", "detector", "csv_name"], sort=True):
    row = metrics_for_group(group)
    metrics.append(row)
    json_path = REPORT_DIR / "metrics" / f"{slugify(row['comparison_id'])}_{slugify(row['subset'])}_{slugify(row['detector'])}_{slugify(row['csv_name'])}.json"
    with json_path.open("w") as f:
        json.dump(row, f, indent=2)

metrics_df = pd.DataFrame(metrics).sort_values(["comparison_id", "subset", "detector", "csv_name"])
metrics_table = metrics_df.drop(columns=["confusion_matrix"])
metrics_table.to_csv(REPORT_DIR / "metrics_table.csv", index=False)
display(metrics_table)

In [ ]:
def plot_confusion(cm, title, output_path):
    fig, ax = plt.subplots(figsize=(4.5, 4.0))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks([0, 1], LABEL_NAMES)
    ax.set_yticks([0, 1], LABEL_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Ground truth")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    return fig


for _, row in metrics_df.iterrows():
    cm = np.array(row["confusion_matrix"])
    name = f"{slugify(row['comparison_id'])}_{slugify(row['subset'])}_{slugify(row['detector'])}_confusion.png"
    title = f"{row['detector']} / {row['comparison_id']} / {row['subset']}"
    fig = plot_confusion(cm, title, REPORT_DIR / "confusion_matrices" / name)
    plt.show()
    plt.close(fig)

## 6. Headline Results Table

In [ ]:
acc_pivot = metrics_df.pivot_table(index=["comparison_id", "test_set", "subset"], columns="detector", values="accuracy", aggfunc="first")
n_by_key = metrics_df.groupby(["comparison_id", "test_set", "subset"])["n"].min().rename("n")
results_table = acc_pivot.rename(columns={"vision": "vision_acc", "tactile": "tactile_acc", "fusion": "fusion_acc"}).join(n_by_key)
if {"tactile_acc", "vision_acc"}.issubset(results_table.columns):
    results_table["tactile_lift"] = results_table["tactile_acc"] - results_table["vision_acc"]
if {"fusion_acc", "vision_acc"}.issubset(results_table.columns):
    results_table["fusion_lift"] = results_table["fusion_acc"] - results_table["vision_acc"]
results_table = results_table.reset_index()
results_table.insert(0, "label", "grasp_outcome")
results_table.to_csv(REPORT_DIR / "results_table.csv", index=False)
display(results_table)

## 7. Object Accuracy Table

In [ ]:
object_accuracy_long = (
    predictions.assign(correct=lambda d: d["label"] == d["pred"])
    .groupby(["comparison_id", "object_id", "is_visually_challenging", "detector"], as_index=False)
    .agg(n=("sample_id", "nunique"), accuracy=("correct", "mean"))
)
object_accuracy = object_accuracy_long.pivot_table(
    index=["comparison_id", "object_id", "is_visually_challenging"], columns="detector", values="accuracy", aggfunc="first"
).reset_index()
detector_cols = [c for c in ["vision", "tactile", "fusion"] if c in object_accuracy.columns]


def best_detectors(row, cols):
    values = row[cols].dropna()
    if values.empty:
        return np.nan
    best = values.max()
    return ",".join(values[values == best].index.tolist())


object_accuracy["best_classifier_by_accuracy"] = object_accuracy.apply(lambda r: best_detectors(r, detector_cols), axis=1)
object_counts = predictions.groupby(["comparison_id", "object_id"])["sample_id"].nunique().rename("n_samples").reset_index()
object_accuracy = object_counts.merge(object_accuracy, on=["comparison_id", "object_id"], how="right")
object_accuracy.to_csv(REPORT_DIR / "object_accuracy_table.csv", index=False)
display(object_accuracy.sort_values(["comparison_id", "is_visually_challenging", "best_classifier_by_accuracy", "object_id"]))

## 8. Object Confidence Table For Correct Predictions

In [ ]:
confidence_rows = predictions.copy()
confidence_rows["correct"] = confidence_rows["label"] == confidence_rows["pred"]
confidence_rows["true_label_confidence"] = np.where(
    confidence_rows["label"] == POSITIVE_LABEL,
    confidence_rows["prob_success"],
    confidence_rows["prob_failure"],
)

object_confidence_long = (
    confidence_rows[confidence_rows["correct"]]
    .groupby(["comparison_id", "object_id", "is_visually_challenging", "detector"], as_index=False)
    .agg(n_correct=("sample_id", "nunique"), mean_correct_confidence=("true_label_confidence", "mean"))
)
object_confidence = object_confidence_long.pivot_table(
    index=["comparison_id", "object_id", "is_visually_challenging"],
    columns="detector",
    values="mean_correct_confidence",
    aggfunc="first",
).reset_index()
confidence_detector_cols = [c for c in ["vision", "tactile", "fusion"] if c in object_confidence.columns]
object_confidence["best_classifier_by_correct_confidence"] = object_confidence.apply(lambda r: best_detectors(r, confidence_detector_cols), axis=1)
object_confidence = object_counts.merge(object_confidence, on=["comparison_id", "object_id"], how="right")
object_confidence.to_csv(REPORT_DIR / "object_correct_confidence_table.csv", index=False)
display(object_confidence.sort_values(["comparison_id", "is_visually_challenging", "best_classifier_by_correct_confidence", "object_id"]))

## 9. Error Slices For The Writeup

In [ ]:
wide = predictions.pivot_table(
    index=["comparison_id", "sample_id", "object_id", "is_visually_challenging", "label"],
    columns="detector",
    values=["pred", "prob_success"],
    aggfunc="first",
)
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

for det in ["vision", "tactile", "fusion"]:
    pred_col = f"pred_{det}"
    if pred_col in wide.columns:
        wide[f"correct_{det}"] = wide[pred_col] == wide["label"]

if {"correct_vision", "correct_tactile"}.issubset(wide.columns):
    touch_catches = wide[(~wide["correct_vision"]) & (wide["correct_tactile"])]
    touch_catches_table = touch_catches.groupby(["comparison_id", "is_visually_challenging", "object_id"]).size().rename("vision_wrong_tactile_correct").reset_index()
    display(touch_catches_table.sort_values("vision_wrong_tactile_correct", ascending=False).head(30))

if {"correct_vision", "correct_fusion"}.issubset(wide.columns):
    fusion_catches = wide[(~wide["correct_vision"]) & (wide["correct_fusion"])]
    fusion_catches_table = fusion_catches.groupby(["comparison_id", "is_visually_challenging", "object_id"]).size().rename("vision_wrong_fusion_correct").reset_index()
    display(fusion_catches_table.sort_values("vision_wrong_fusion_correct", ascending=False).head(30))

vision_counter_slices = []
if {"correct_vision", "correct_tactile"}.issubset(wide.columns):
    table = wide[(wide["correct_vision"]) & (~wide["correct_tactile"])].groupby(["comparison_id", "is_visually_challenging", "object_id"]).size().rename("vision_correct_tactile_wrong").reset_index()
    vision_counter_slices.append(table)
if {"correct_vision", "correct_fusion"}.issubset(wide.columns):
    table = wide[(wide["correct_vision"]) & (~wide["correct_fusion"])].groupby(["comparison_id", "is_visually_challenging", "object_id"]).size().rename("vision_correct_fusion_wrong").reset_index()
    vision_counter_slices.append(table)
if {"correct_vision", "correct_tactile", "correct_fusion"}.issubset(wide.columns):
    table = wide[(wide["correct_vision"]) & (~wide["correct_tactile"]) & (~wide["correct_fusion"])].groupby(["comparison_id", "is_visually_challenging", "object_id"]).size().rename("vision_correct_tactile_and_fusion_wrong").reset_index()
    vision_counter_slices.append(table)

if vision_counter_slices:
    vision_counter_table = object_counts.merge(
        predictions[["comparison_id", "object_id", "is_visually_challenging"]].drop_duplicates(),
        on=["comparison_id", "object_id"],
        how="left",
    )
    for table in vision_counter_slices:
        vision_counter_table = vision_counter_table.merge(table, on=["comparison_id", "is_visually_challenging", "object_id"], how="left")
    count_cols = [c for c in vision_counter_table.columns if c.startswith("vision_correct_")]
    vision_counter_table[count_cols] = vision_counter_table[count_cols].fillna(0).astype(int)
    vision_counter_table["any_vision_counter_case"] = vision_counter_table[count_cols].sum(axis=1)
    vision_counter_table = vision_counter_table.sort_values(["comparison_id", "any_vision_counter_case", "object_id"], ascending=[True, False, True])
    vision_counter_table.to_csv(REPORT_DIR / "objects_vision_caught_touch_or_fusion_missed.csv", index=False)
    display(vision_counter_table[vision_counter_table["any_vision_counter_case"] > 0])

wide.to_csv(REPORT_DIR / "sample_level_wide_predictions.csv", index=False)